# Face Recognition Model Training Pipeline
**Deep Learning 128D Embeddings + SVM/KNN Classification**

Pipeline: Upload Dataset > Preprocess > Augment > Encode Faces > Train > Evaluate > Download Models

In [ ]:
# Cell 1: Install Dependencies (takes ~3-4 minutes)
# dlib compilation takes time on Colab - be patient!
!pip install -q --upgrade face_recognition albumentations scikit-learn matplotlib seaborn tqdm imutils
print("")
print("All dependencies installed!")
import albumentations; print(f"   albumentations version: {albumentations.__version__}")

## Upload Your Dataset
Upload your **datasetAttendence.zip** file. Structure should be:
```
datasetAttendence/
  aniket/  (img1.jpeg, img2.jpeg, ...)
  rohan/   (img1.jpeg, img2.jpeg, ...)
  yash/    (img1.jpeg, img2.jpeg, ...)
```

In [ ]:
# Cell 2: Upload and Extract Dataset
import zipfile, os, shutil
from google.colab import files

# Clean previous extractions
if os.path.exists("dataset_upload"):
    shutil.rmtree("dataset_upload")

print("Select your dataset zip file:")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f"Extracting {zip_name}...")
with zipfile.ZipFile(zip_name, 'r') as z:
    # Check if zip has folder structure in paths
    names = z.namelist()
    has_folders = any('/' in n for n in names)

    if has_folders:
        # Extract preserving internal folder structure
        for member in names:
            # Ensure parent dirs exist
            target = os.path.join("dataset_upload", member)
            if member.endswith('/'):
                os.makedirs(target, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(target), exist_ok=True)
                with z.open(member) as src, open(target, 'wb') as dst:
                    dst.write(src.read())
    else:
        z.extractall("dataset_upload")

# Debug: show extracted structure
print("")
print("Extracted contents:")
for root, dirs, fls in os.walk("dataset_upload"):
    level = root.replace("dataset_upload", "").count(os.sep)
    indent = "  " * level
    base = os.path.basename(root)
    print(f"{indent} {base}/  ({len(fls)} files, {len(dirs)} subdirs)")
    if level >= 3:
        break

# Find the folder containing student subfolders with images
DATASET_PATH = None
for root, dirs, fls in os.walk("dataset_upload"):
    student_dirs = []
    for d in dirs:
        subdir = os.path.join(root, d)
        has_images = any(
            f.lower().endswith(('.jpg', '.jpeg', '.png'))
            for f in os.listdir(subdir)
            if os.path.isfile(os.path.join(subdir, f))
        )
        if has_images:
            student_dirs.append(d)
    if len(student_dirs) >= 2:
        DATASET_PATH = root
        break

if DATASET_PATH is None:
    print("")
    print("ERROR - Could not auto-detect dataset. Listing all folders:")
    for root, dirs, fls in os.walk("dataset_upload"):
        print(f"  {root}/ -> subdirs={dirs}, files={len(fls)}")
    raise FileNotFoundError(
        "No valid dataset found! "
        "Zip should contain folders like: student_name1/img.jpg, student_name2/img.jpg"
    )

students = [d for d in sorted(os.listdir(DATASET_PATH))
            if os.path.isdir(os.path.join(DATASET_PATH, d)) and not d.startswith('.')]
students = [s for s in students
            if any(f.lower().endswith(('.jpg','.jpeg','.png'))
                   for f in os.listdir(os.path.join(DATASET_PATH, s)))]

print("")
print(f"Dataset found at: {DATASET_PATH}")
print(f"Students found: {len(students)}")
for s in students:
    imgs = [f for f in os.listdir(os.path.join(DATASET_PATH, s))
            if f.lower().endswith(('.jpg','.jpeg','.png'))]
    print(f"   {s}: {len(imgs)} images")

In [ ]:
# Cell 3: Imports and Configuration
import cv2
import numpy as np
import face_recognition
import pickle, json, os, logging, time, warnings
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns
import albumentations as A
import joblib
warnings.filterwarnings('ignore')

# === CONFIGURATION ===
BLUR_THRESHOLD = 35.0
MIN_BRIGHTNESS = 30.0
MAX_BRIGHTNESS = 250.0
MIN_FACE_SIZE = 30
IMAGE_RESIZE_WIDTH = 600
DETECTION_MODEL = "hog"
NUM_JITTERS = 5
ENCODING_MODEL = "large"
NUM_AUGMENTATIONS = 5
OUTLIER_STD_THRESHOLD = 2.0
DUPLICATE_THRESHOLD = 0.98
TEST_SPLIT_RATIO = 0.2
CV_FOLDS = 5
SEED = 42
CONFIDENCE_THRESHOLD = 0.55

# Output dirs
os.makedirs("models", exist_ok=True)
os.makedirs("encodings", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print("Configuration loaded!")
print(f"   Detection: {DETECTION_MODEL} | Jitters: {NUM_JITTERS} | Augmentations: {NUM_AUGMENTATIONS}")

In [ ]:
# Cell 4: Image Preprocessing

def load_image(path):
    """Safely load image, handle special characters in filenames."""
    try:
        data = np.fromfile(str(path), dtype=np.uint8)
        img = cv2.imdecode(data, cv2.IMREAD_COLOR)
        return img if img is not None and img.size > 0 else None
    except Exception:
        return None

def check_blur(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def check_brightness(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    return float(hsv[:,:,2].mean())

def resize_image(img, width=IMAGE_RESIZE_WIDTH):
    h, w = img.shape[:2]
    if w == width:
        return img
    scale = width / w
    interp = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    return cv2.resize(img, (width, int(h * scale)), interpolation=interp)

def normalize_illumination(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)

def preprocess_dataset(dataset_path):
    valid = []
    skipped = 0
    students = sorted([d for d in os.listdir(dataset_path)
                       if os.path.isdir(os.path.join(dataset_path, d)) and not d.startswith('.')])

    for student in students:
        folder = os.path.join(dataset_path, student)
        imgs = [f for f in sorted(os.listdir(folder))
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        for fname in tqdm(imgs, desc=f"  {student}", leave=False):
            fpath = os.path.join(folder, fname)
            img = load_image(fpath)
            if img is None:
                skipped += 1; continue

            img = resize_image(img)

            # Quality checks
            blur = check_blur(img)
            if blur < BLUR_THRESHOLD:
                skipped += 1; continue

            bright = check_brightness(img)
            if not (MIN_BRIGHTNESS <= bright <= MAX_BRIGHTNESS):
                skipped += 1; continue

            img = normalize_illumination(img)
            valid.append((img, student, fname))

    print("")
    print(f"Preprocessing complete: {len(valid)} valid, {skipped} skipped")
    for name in students:
        count = sum(1 for _, l, _ in valid if l == name)
        print(f"   {name}: {count} images")
    return valid

# --- RUN PREPROCESSING ---
print("=" * 50)
print("STEP 1: IMAGE PREPROCESSING")
print("=" * 50)
valid_images = preprocess_dataset(DATASET_PATH)

In [ ]:
# Cell 5: Data Augmentation

aug_pipeline = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.8),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, border_mode=cv2.BORDER_REFLECT_101, p=0.6),
    A.GaussNoise(std_range=(0.02, 0.08), p=0.4),
    A.GaussianBlur(blur_limit=(3, 3), p=0.3),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.4),
    A.CLAHE(clip_limit=2.0, p=0.3),
])

def augment_dataset(images_with_labels):
    augmented = list(images_with_labels)
    aug_count = 0

    for img, label, fname in tqdm(images_with_labels, desc="Augmenting"):
        for i in range(NUM_AUGMENTATIONS):
            try:
                aug = aug_pipeline(image=img)["image"]
                if aug is not None and aug.size > 0:
                    augmented.append((aug, label, f"{Path(fname).stem}_aug{i}{Path(fname).suffix}"))
                    aug_count += 1
            except Exception:
                pass

    print("")
    print(f"Augmentation complete: {len(images_with_labels)} original + {aug_count} augmented = {len(augmented)} total")
    labels_list = [l for _, l, _ in augmented]
    for name in sorted(set(labels_list)):
        print(f"   {name}: {labels_list.count(name)} images")
    return augmented

# --- RUN AUGMENTATION ---
print("=" * 50)
print("STEP 2: DATA AUGMENTATION")
print("=" * 50)
augmented_images = augment_dataset(valid_images)

In [ ]:
# Cell 6: Face Encoding Generation (128D dlib embeddings)

def encode_dataset(images_with_labels):
    encodings, labels, filenames = [], [], []
    skip = {"no_face": 0, "multi_face": 0, "small": 0, "error": 0}

    for img, label, fname in tqdm(images_with_labels, desc="Encoding faces"):
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Detect face
        locs = face_recognition.face_locations(rgb, model=DETECTION_MODEL)
        if len(locs) == 0:
            skip["no_face"] += 1; continue
        if len(locs) > 1:
            skip["multi_face"] += 1; continue

        top, right, bottom, left = locs[0]
        if (right - left) < MIN_FACE_SIZE or (bottom - top) < MIN_FACE_SIZE:
            skip["small"] += 1; continue

        # Generate 128D embedding
        try:
            encs = face_recognition.face_encodings(rgb, locs, num_jitters=NUM_JITTERS, model=ENCODING_MODEL)
            if encs:
                encodings.append(encs[0])
                labels.append(label)
                filenames.append(fname)
        except Exception:
            skip["error"] += 1

    print("")
    print(f"Encoding complete: {len(encodings)} successful")
    for k, v in skip.items():
        if v > 0: print(f"   Skipped ({k}): {v}")

    if not encodings:
        raise ValueError("No face encodings generated! Check your dataset images.")

    # Remove duplicates within each class
    enc_clean, lab_clean, fn_clean = [], [], []
    removed = 0
    label_groups = {}
    for i, l in enumerate(labels):
        label_groups.setdefault(l, []).append(i)

    for label, indices in label_groups.items():
        class_enc = [encodings[i] for i in indices]
        class_fn = [filenames[i] for i in indices]
        keep = [True] * len(class_enc)
        for i in range(len(class_enc)):
            if not keep[i]: continue
            for j in range(i+1, len(class_enc)):
                if not keep[j]: continue
                sim = np.dot(class_enc[i], class_enc[j]) / (np.linalg.norm(class_enc[i]) * np.linalg.norm(class_enc[j]))
                if sim > DUPLICATE_THRESHOLD:
                    keep[j] = False; removed += 1
        for i, k in enumerate(keep):
            if k:
                enc_clean.append(class_enc[i])
                lab_clean.append(label)
                fn_clean.append(class_fn[i])

    if removed: print(f"   Removed {removed} duplicate embeddings")

    # Remove outliers
    enc_final, lab_final, fn_final = [], [], []
    outliers = 0
    label_groups2 = {}
    for i, l in enumerate(lab_clean):
        label_groups2.setdefault(l, []).append(i)

    for label, indices in label_groups2.items():
        ce = np.array([enc_clean[i] for i in indices])
        cf = [fn_clean[i] for i in indices]
        if len(ce) < 3:
            for i, idx in enumerate(indices):
                enc_final.append(enc_clean[idx]); lab_final.append(label); fn_final.append(cf[i])
            continue
        centroid = ce.mean(axis=0)
        dists = np.linalg.norm(ce - centroid, axis=1)
        threshold = dists.mean() + OUTLIER_STD_THRESHOLD * dists.std()
        for i, idx in enumerate(indices):
            if dists[i] <= threshold:
                enc_final.append(enc_clean[idx]); lab_final.append(label); fn_final.append(cf[i])
            else:
                outliers += 1

    if outliers: print(f"   Removed {outliers} outlier embeddings")

    if not enc_final:
        raise ValueError("No valid face encodings after cleaning! Check your images.")

    # L2 normalize
    enc_array = normalize(np.array(enc_final), norm='l2')
    print("")
    print(f"Final: {enc_array.shape[0]} encodings, shape={enc_array.shape}")
    for name in sorted(set(lab_final)):
        print(f"   {name}: {lab_final.count(name)} encodings")

    # Save
    with open("encodings/encodings.pkl", "wb") as f:
        pickle.dump({"encodings": enc_array, "labels": lab_final, "filenames": fn_final}, f)
    print(f"Saved to encodings/encodings.pkl")

    return enc_array, lab_final, fn_final

# --- RUN ENCODING ---
print("=" * 50)
print("STEP 3: FACE ENCODING GENERATION")
print("=" * 50)
t = time.time()
encodings, labels, filenames = encode_dataset(augmented_images)
print(f"Encoding took {time.time()-t:.1f}s")

In [ ]:
# Cell 7: Model Training (SVM + KNN with GridSearchCV)

def train_models(encodings, labels):
    le = LabelEncoder()
    y = le.fit_transform(labels)

    # Smart split
    unique, counts = np.unique(y, return_counts=True)
    min_count = counts.min()
    stratify = y if min_count >= 2 else None
    test_size = TEST_SPLIT_RATIO if len(encodings) >= 20 else max(0.15, 1.0/len(unique))

    X_train, X_test, y_train, y_test = train_test_split(
        encodings, y, test_size=test_size, random_state=SEED, stratify=stratify)

    print(f"Train: {len(X_train)} | Test: {len(X_test)} | Classes: {list(le.classes_)}")

    # CV folds
    min_class_train = min(np.sum(y_train == c) for c in unique)
    cv = min(CV_FOLDS, min_class_train, len(X_train))
    cv = max(2, cv)

    # --- SVM ---
    print(f"")
    print(f"Training SVM (GridSearchCV, {cv}-fold CV)...")
    svm_grid = GridSearchCV(
        SVC(probability=True, random_state=SEED),
        {"C": [0.1, 1, 10, 100], "gamma": ["scale", "auto", 0.001, 0.01], "kernel": ["rbf", "linear"]},
        cv=StratifiedKFold(cv, shuffle=True, random_state=SEED),
        scoring="accuracy", n_jobs=-1, refit=True)
    svm_grid.fit(X_train, y_train)
    svm_model = svm_grid.best_estimator_
    print(f"   Best params: {svm_grid.best_params_}")
    print(f"   Best CV accuracy: {svm_grid.best_score_*100:.2f}%")

    # --- KNN ---
    max_k = min(7, len(X_train) - 1)
    k_values = [k for k in [1, 3, 5, 7] if k <= max_k] or [1]
    print(f"")
    print(f"Training KNN (GridSearchCV, {cv}-fold CV)...")
    knn_grid = GridSearchCV(
        KNeighborsClassifier(),
        {"n_neighbors": k_values, "weights": ["uniform", "distance"], "metric": ["cosine", "euclidean"]},
        cv=StratifiedKFold(cv, shuffle=True, random_state=SEED),
        scoring="accuracy", n_jobs=-1, refit=True)
    knn_grid.fit(X_train, y_train)
    knn_model = knn_grid.best_estimator_
    print(f"   Best params: {knn_grid.best_params_}")
    print(f"   Best CV accuracy: {knn_grid.best_score_*100:.2f}%")

    # --- Threshold optimization ---
    print("")
    print("Optimizing confidence threshold...")
    best_thresh, best_score = CONFIDENCE_THRESHOLD, 0
    probas = svm_model.predict_proba(X_test)
    preds = svm_model.predict(X_test)
    for t in np.arange(0.3, 0.91, 0.05):
        mask = probas.max(axis=1) >= t
        if mask.sum() == 0: continue
        acc = accuracy_score(y_test[mask], preds[mask])
        score = acc * 0.7 + mask.mean() * 0.3
        if score > best_score:
            best_score = score; best_thresh = round(t, 2)
    print(f"   Optimal threshold: {best_thresh}")

    # Save models
    joblib.dump(svm_model, "models/svm_model.pkl")
    joblib.dump(knn_model, "models/knn_model.pkl")
    joblib.dump(le, "models/label_encoder.pkl")

    best_name = "SVM" if svm_grid.best_score_ >= knn_grid.best_score_ else "KNN"
    best_m = svm_model if best_name == "SVM" else knn_model
    joblib.dump(best_m, "models/best_model.pkl")

    with open("models/threshold.json", "w") as f:
        json.dump({"threshold": best_thresh, "best_model": best_name,
                    "svm_cv": round(svm_grid.best_score_,4), "knn_cv": round(knn_grid.best_score_,4)}, f, indent=2)

    print(f"")
    print(f"Models saved! Best: {best_name}")
    return svm_model, knn_model, le, X_train, X_test, y_train, y_test, best_thresh

# --- RUN TRAINING ---
print("=" * 50)
print("STEP 4: MODEL TRAINING")
print("=" * 50)
t = time.time()
svm_model, knn_model, le, X_train, X_test, y_train, y_test, best_thresh = train_models(encodings, labels)
print(f"Training took {time.time()-t:.1f}s")

In [ ]:
# Cell 8: Evaluation and Visualization

def evaluate(model, name, X_test, y_test, le):
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average="weighted", zero_division=0)
    rec = recall_score(y_test, preds, average="weighted", zero_division=0)
    f1 = f1_score(y_test, preds, average="weighted", zero_division=0)

    print("")
    print(f"--- {name} Results ---")
    print(f"  Accuracy : {acc*100:.2f}%")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1 Score : {f1:.4f}")
    print("")
    print(classification_report(y_test, preds, target_names=le.classes_, zero_division=0))
    return acc, prec, rec, f1, preds

print("=" * 50)
print("STEP 5: MODEL EVALUATION")
print("=" * 50)

s_acc, s_p, s_r, s_f1, s_preds = evaluate(svm_model, "SVM", X_test, y_test, le)
k_acc, k_p, k_r, k_f1, k_preds = evaluate(knn_model, "KNN", X_test, y_test, le)

# --- Confusion Matrices ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, preds, name in [(axes[0], s_preds, "SVM"), (axes[1], k_preds, "KNN")]:
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=le.classes_,
                yticklabels=le.classes_, ax=ax, linewidths=0.5)
    ax.set_title(f"Confusion Matrix - {name}", fontweight="bold")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.savefig("outputs/confusion_matrices.png", dpi=150); plt.show()

# --- Metrics Comparison ---
fig, ax = plt.subplots(figsize=(10, 5))
metrics = ["Accuracy", "Precision", "Recall", "F1"]
x = np.arange(len(metrics)); w = 0.35
b1 = ax.bar(x-w/2, [s_acc, s_p, s_r, s_f1], w, label="SVM", color="#4C72B0")
b2 = ax.bar(x+w/2, [k_acc, k_p, k_r, k_f1], w, label="KNN", color="#DD8452")
for b in [b1, b2]:
    for bar in b:
        ax.annotate(f"{bar.get_height():.3f}", xy=(bar.get_x()+bar.get_width()/2, bar.get_height()),
                     xytext=(0, 4), textcoords="offset points", ha="center", fontsize=9, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(metrics); ax.set_ylim(0, 1.15); ax.legend()
ax.set_title("SVM vs KNN Comparison", fontweight="bold"); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig("outputs/training_metrics.png", dpi=150); plt.show()

# --- Confidence Distribution ---
if hasattr(svm_model, "predict_proba"):
    probas = svm_model.predict_proba(X_test).max(axis=1)
    correct = s_preds == y_test
    fig, ax = plt.subplots(figsize=(8, 4))
    if correct.sum(): ax.hist(probas[correct], bins=15, alpha=0.7, label="Correct", color="#2ecc71")
    if (~correct).sum(): ax.hist(probas[~correct], bins=15, alpha=0.7, label="Incorrect", color="#e74c3c")
    ax.set_xlabel("Confidence"); ax.set_title("Confidence Distribution", fontweight="bold"); ax.legend()
    plt.tight_layout(); plt.savefig("outputs/confidence_dist.png", dpi=150); plt.show()

best = "SVM" if s_acc >= k_acc else "KNN"
print("")
print("=" * 50)
print(f"BEST MODEL: {best}")
print(f"SVM Accuracy: {s_acc*100:.2f}% | KNN Accuracy: {k_acc*100:.2f}%")
print(f"Confidence Threshold: {best_thresh}")
print("=" * 50)

In [ ]:
# Cell 9: Download Trained Models

import shutil
from google.colab import files

# Create zip with all outputs
shutil.make_archive("trained_models", "zip", ".", "models")
shutil.make_archive("all_outputs", "zip", ".", "outputs")
shutil.make_archive("face_encodings", "zip", ".", "encodings")

print("Files ready for download:")
print("   1. trained_models.zip (SVM, KNN, LabelEncoder, threshold)")
print("   2. face_encodings.zip (128D embeddings)")
print("   3. all_outputs.zip (plots)")

print("")
print("Downloading trained_models.zip...")
files.download("trained_models.zip")

print("Downloading face_encodings.zip...")
files.download("face_encodings.zip")

print("Downloading all_outputs.zip...")
files.download("all_outputs.zip")